In [1]:
import torch
import torchmetrics
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torchvision
import torchvision.transforms.v2 as T

In [2]:
train_and_valid_data = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True, 
    download=True,
    # transform
)
test_data = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False, 
    download=True,
    # transform
)
train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [0.9, 0.1])
print(
    len(train_data), 
    len(valid_data), 
    len(test_data), 
    )

54000 6000 10000


In [4]:
img, label = train_and_valid_data[1]
img

In [5]:
# toImage = T.ToImage()
# toDtype = T.ToDtype(dtype=torch.float32, scale=True)
# toDtype(toImage(img))

toTensor = T.Compose([
    T.ToImage(), 
    # T.Resize((28,28)), 
    T.ToDtype(dtype=torch.float32, scale=True)
])
toTensor(img)

Image([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.0000,
         0.0000, 0.1608, 0.7373, 0.4039, 0.2118, 0.1882, 0.1686, 0.3412, 0.6588,
         0.5216, 0.0627, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.0000, 0.1922, 0.5333,
         0.8588, 0.8471, 0.8941, 0.9255, 1.0000, 1.0000, 1.0000, 1.0000, 0.8510,
         0.8431, 0.9961, 0.9059, 0.6275, 0.1765, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0549, 0.6902, 0.8706, 0.8784,
         0.8314, 0.7961, 0.7765, 0.7686, 0.7843, 0.8431, 0.8000, 0.7922, 0.7882,
         0.7882, 0.7882, 0.8196, 0.8549, 0.8784, 0.6431, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.7373, 0.8588, 0.7843, 0.7765,
         0.7922, 0.7765, 0.7804, 0.7804, 0.7882, 0.7686, 0.7765, 0.7765, 0.7843,
         0.7843, 0.7843, 0.7843, 0.7882, 0.7843, 0.8824

In [6]:
train_and_valid_data = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True, 
    download=True,
    transform = toTensor,
)
test_data = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False, 
    download=True,
    transform = toTensor,
)
train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [0.9, 0.1])
print(
    len(train_data), 
    len(valid_data), 
    len(test_data), 
    )

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

54000 6000 10000


In [7]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cpu'

In [8]:
54000/32

1687.5

In [9]:
def train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
	history = {
		'loss' : [],
		'train_metric' : [],
		'valid_metric' : [],
	}
	for epoch in range(n_epochs):
		# Training 
		total_loss = 0
		metric.reset()
		for X_batch, y_batch in train_loader:
			X_batch, y_batch = X_batch.to(device), y_batch.to(device)
			model.train()
			y_pred = model(X_batch)
			loss = criterion(y_pred, y_batch)
			total_loss += loss.item()
			loss.backward()
			optimizer.step()
			# for group in optimizer.param_groups:
			# 	for p in group['params']:
			# 		if p.grad is not None:
			# 			print(f'\t{p.grad.max()}') 
			optimizer.zero_grad()
			metric.update(y_pred, y_batch)
		
		avg_loss = total_loss / len(train_loader)
		history['loss'].append(avg_loss)

		avg_metric_train = metric.compute().item()
		history['train_metric'].append(avg_metric_train)

		# Evaluation 
		model.eval()
		metric.reset()
		with torch.no_grad():
			for X_batch, y_batch in valid_loader:
				X_batch, y_batch = X_batch.to(device), y_batch.to(device)
				y_pred = model(X_batch)
				metric.update(y_pred, y_batch)

		avg_metric_valid = metric.compute().item()
		history['valid_metric'].append(avg_metric_valid)

		print(
			f'Epoch: {epoch+1}/{n_epochs}, '
			+f'Loss: {round(avg_loss,3)}, '
			+f'Train Metric: {round(avg_metric_train,3)}, ' 
			+f'Valid Metric: {round(avg_metric_valid,3)}'
		)

		# if epoch>=2:
		# 	break
	return history

def plot_history(history, n_epochs, metric):
    plt.plot(np.arange(n_epochs) + 1, history['train_metric'], linestyle='--', color='r', marker='.', label='Train')
    plt.plot(np.arange(n_epochs) + 1, history['valid_metric'], linestyle='--', color='b', marker='.', label='Valid')
    plt.legend()
    plt.grid()
    plt.xlabel('Epochs')
    plt.ylabel(f'{metric.__class__.__name__}')
    plt.show()

In [23]:
criterion = nn.CrossEntropyLoss()

y_pred_logit = torch.tensor([
    [-1.23, 1.28, 0.5], 
    [2.2, -4.4, 3.7,]
])
y_train = torch.tensor([1, 0])

y_prob = y_pred_logit.softmax(dim=1).round(decimals=3)
y_prob

tensor([[0.0530, 0.6490, 0.2980],
        [0.1820, 0.0000, 0.8170]])

In [26]:
correct_class_probs = y_prob[[0,1], y_train]
correct_class_probs

tensor([0.6490, 0.1820])

In [29]:
-torch.log(correct_class_probs).mean()

tensor(1.0680)

In [24]:
criterion = nn.CrossEntropyLoss()
criterion(y_pred_logit, y_train)

tensor(1.0666)

In [39]:
train_loader.dataset[0][0].shape

torch.Size([1, 28, 28])

In [42]:
# arr = np.array([
#     [1,2,3],
#     [5,6,7]
# ])

# arr.flatten()

In [43]:
n_epochs=20

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features=1*28*28, out_features=200), 
    nn.ReLU(),
    nn.Linear(in_features=200, out_features=100), 
    nn.ReLU(),
    nn.Linear(in_features=100, out_features=10)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.1)
metric = torchmetrics.Accuracy(task='multiclass', num_classes=10)

train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs)

Epoch: 1/20, Loss: 0.611, Train Metric: 0.779, Valid Metric: 0.817
Epoch: 2/20, Loss: 0.41, Train Metric: 0.85, Valid Metric: 0.832
Epoch: 3/20, Loss: 0.368, Train Metric: 0.864, Valid Metric: 0.81
Epoch: 4/20, Loss: 0.339, Train Metric: 0.874, Valid Metric: 0.873
Epoch: 5/20, Loss: 0.319, Train Metric: 0.881, Valid Metric: 0.868
Epoch: 6/20, Loss: 0.305, Train Metric: 0.885, Valid Metric: 0.879
Epoch: 7/20, Loss: 0.289, Train Metric: 0.892, Valid Metric: 0.888
Epoch: 8/20, Loss: 0.278, Train Metric: 0.895, Valid Metric: 0.883
Epoch: 9/20, Loss: 0.267, Train Metric: 0.898, Valid Metric: 0.876
Epoch: 10/20, Loss: 0.257, Train Metric: 0.903, Valid Metric: 0.881
Epoch: 11/20, Loss: 0.25, Train Metric: 0.904, Valid Metric: 0.892


KeyboardInterrupt: 